# Statistical Testing for Structural Similarity Evaluation

This notebook computes uncertainty estimates and paired significance tests for the evaluation tasks used in the project: pairwise separation, retrieval, ranking, synthetic anchor comparison, SemEval 2026 triplets, and SemEval 2022 narrative-similarity correlation.

The key principle is that all uncertainty estimates are computed at the example level: pairs are resampled as pairs, retrieval/ranking examples are resampled as whole queries with all options kept together, and SemEval triplets/article pairs are resampled as intact examples. This avoids incorrectly treating multiple scores inside the same query as independent observations.

In [16]:
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModel
except Exception as e:
    torch = None
    F = None
    AutoTokenizer = None
    AutoModel = None
    print('[warn] Torch/transformers unavailable. You can still run stats if cached per-example outputs exist.')
    print('[warn]', e)

from scipy import stats
try:
    from statsmodels.stats.contingency_tables import mcnemar as statsmodels_mcnemar
except Exception:
    statsmodels_mcnemar = None

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'src' else cwd
print('PROJECT_ROOT:', PROJECT_ROOT)

RANDOM_SEED = 42
N_BOOT = 10_000
ALPHA = 0.05
FORCE_RECOMPUTE = False
INCLUDE_EXISTING_LLM_OUTPUTS = False

EVAL_DATA_DIR = PROJECT_ROOT / 'data' / 'eval_data'
EVAL_RESULTS_DIR = PROJECT_ROOT / 'data' / 'eval_results'
OUT_DIR = EVAL_RESULTS_DIR / 'statistical_testing'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PER_EXAMPLE_PARQUET = OUT_DIR / 'embedding_per_example_outputs_epoch02_mse_plus_baselines.parquet'
PER_EXAMPLE_CSV = OUT_DIR / 'embedding_per_example_outputs_epoch02_mse_plus_baselines.csv'
POINT_TABLE_PATH = OUT_DIR / 'metric_bootstrap_ci_table.csv'
TEST_TABLE_PATH = OUT_DIR / 'paired_significance_tests.csv'
SUMMARY_TABLE_PATH = OUT_DIR / 'statistical_testing_summary.csv'

PATHS = {
    'eval_pair': EVAL_DATA_DIR / 'eval_story_pairs_200.csv',
    'retrieval': EVAL_DATA_DIR / 'retrieval_eval_df.csv',
    'ranking': EVAL_DATA_DIR / 'ranking_eval_df.csv',
    'synthetic_v2': EVAL_DATA_DIR / 'synthetic_v2_anchor_eval_data.csv',
    'semeval_2026': EVAL_DATA_DIR / 'SemEval2026-Task_4-dev-v1' / 'dev_track_a.jsonl',
    'semeval_2022': EVAL_DATA_DIR / 'SemEval2022-Task8' / 'semeval_2022_eval_data.csv',
}
for name, path in PATHS.items():
    print(f'{name:15s}: {path} | exists={path.exists()}')

PROJECT_ROOT: /tank/scratch/shayan/Projects/NarrativeSimilarity
eval_pair      : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv | exists=True
retrieval      : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/retrieval_eval_df.csv | exists=True
ranking        : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/ranking_eval_df.csv | exists=True
synthetic_v2   : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/synthetic_v2_anchor_eval_data.csv | exists=True
semeval_2026   : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl | exists=True
semeval_2022   : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/SemEval2022-Task8/semeval_2022_eval_data.csv | exists=True


## Model and Output Configuration

The statistical tests need per-example outputs from every model being compared. If cached per-example outputs already exist, the notebook loads them directly. Otherwise, it reconstructs them by loading the trained checkpoint and embedding baselines, running the same evaluation datasets, and saving a reusable cache.

The primary system is labeled `E5-Mistral+MSE`. Edit `OUR_CHECKPOINT_CANDIDATES` if the checkpoint lives somewhere else on a given machine.

In [19]:
OUR_MODEL_NAME = 'E5-Mistral+MSE'
OUR_CHECKPOINT = Path('/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss/epoch_01_mse_loss')
print('OUR_CHECKPOINT:', OUR_CHECKPOINT)
if not OUR_CHECKPOINT.exists():
    raise FileNotFoundError(f'Target checkpoint not found: {OUR_CHECKPOINT}')

HF_CACHE_CANDIDATES = [
    PROJECT_ROOT / 'hf_cache',
    Path('/scratch/shayan/hf_cache'),
    Path('/tank/scratch/shayan/hf_cache'),
]
HF_CACHE_DIR = next((p for p in HF_CACHE_CANDIDATES if p.exists()), PROJECT_ROOT / 'hf_cache')
print('HF_CACHE_DIR:', HF_CACHE_DIR)

# Restrict trained-model evaluation to the requested checkpoint, but include
# embedding baselines so paired significance tests have comparison rows.
MODEL_SPECS = [
    {
        'model': OUR_MODEL_NAME,
        'model_type': 'checkpoint',
        'path': OUR_CHECKPOINT,
        'use_e5_query_prefix': True,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'bge_base',
        'model_type': 'embedding_baseline',
        'path': 'BAAI/bge-base-en-v1.5',
        'use_e5_query_prefix': False,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'e5_untrained',
        'model_type': 'embedding_baseline',
        'path': 'intfloat/e5-mistral-7b-instruct',
        'use_e5_query_prefix': True,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'roberta_base',
        'model_type': 'embedding_baseline',
        'path': 'roberta-base',
        'use_e5_query_prefix': False,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'story_emb',
        'model_type': 'embedding_baseline',
        'path': 'uhhlt/story-emb',
        'use_e5_query_prefix': False,
        'max_length': 1024,
        'use_bfloat16': True,
    },
]

pd.DataFrame([{k: str(v) for k, v in spec.items()} for spec in MODEL_SPECS])


OUR_CHECKPOINT: /scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss/epoch_01_mse_loss
HF_CACHE_DIR: /scratch/shayan/hf_cache


,model,model_type,path,use_e5_query_prefix,max_length,use_bfloat16
0,E5-Mistral+MSE,checkpoint,/scratch/shayan/Projects/NarrativeSimilarity/a...,True,1024,True
1,bge_base,embedding_baseline,BAAI/bge-base-en-v1.5,False,1024,True
2,e5_untrained,embedding_baseline,intfloat/e5-mistral-7b-instruct,True,1024,True
3,roberta_base,embedding_baseline,roberta-base,False,1024,True
4,story_emb,embedding_baseline,uhhlt/story-emb,False,1024,True


## Reconstruct Per-Example Outputs

This cell mirrors the evaluation logic from the CLI script, but stores one row per evaluation example instead of only aggregate metrics. These rows are what bootstrap confidence intervals and paired tests require.

The output cache is saved under `data/eval_results/statistical_testing/`. If the cache exists, set `FORCE_RECOMPUTE = False` to load it immediately.

In [20]:
def read_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def parse_bool_label(v) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, np.integer)):
        return bool(v)
    if isinstance(v, str):
        vv = v.strip().lower()
        if vv in {'true', '1', 'yes', 'y', 'a', 'text_a', 'text_a_is_closer'}:
            return True
        if vv in {'false', '0', 'no', 'n', 'b', 'text_b'}:
            return False
    raise ValueError(f'Could not parse boolean label: {v!r}')


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom


class Embedder:
    def __init__(self, model_path, max_length=1024, use_bfloat16=True, use_e5_query_prefix=True):
        if torch is None or AutoTokenizer is None or AutoModel is None:
            raise RuntimeError('Torch/transformers are required to recompute embedding outputs.')
        if model_path is None:
            raise FileNotFoundError('Checkpoint path is None. Update OUR_CHECKPOINT_CANDIDATES or load an existing cache.')

        self.model_path = str(model_path)
        self.use_e5_query_prefix = bool(use_e5_query_prefix)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        dtype = torch.bfloat16 if (self.device == 'cuda' and use_bfloat16) else torch.float32

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path,
            trust_remote_code=True,
            cache_dir=str(HF_CACHE_DIR),
        )
        self.model = AutoModel.from_pretrained(
            self.model_path,
            trust_remote_code=True,
            cache_dir=str(HF_CACHE_DIR),
            dtype=dtype,
        )
        self.model.to(self.device)
        self.model.eval()

        tokenizer_cap = getattr(self.tokenizer, 'model_max_length', None)
        config_cap = getattr(getattr(self.model, 'config', None), 'max_position_embeddings', None)
        caps = [int(max_length)]
        if isinstance(tokenizer_cap, int) and 0 < tokenizer_cap < 10**6:
            caps.append(tokenizer_cap)
        if isinstance(config_cap, int) and 0 < config_cap < 10**6:
            caps.append(config_cap)
        self.max_length = int(min(caps))
        print(f'[Embedder] {self.model_path} | device={self.device} | max_length={self.max_length} | e5_prefix={self.use_e5_query_prefix}')

    def prepare_text(self, text: str) -> str:
        text = str(text)
        if self.use_e5_query_prefix:
            return f'query: retrieve stories with a similar narrative to the given story. {text}'
        return text

    @torch.no_grad()
    def embed(self, texts, batch_size=8):
        vectors = []
        for i in range(0, len(texts), batch_size):
            batch = list(texts[i:i + batch_size])
            enc = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}
            h = self.model(**enc).last_hidden_state
            pooled = mean_pool(h, enc['attention_mask'])
            pooled = F.normalize(pooled, p=2, dim=1)
            vectors.append(pooled.detach().float().cpu())
        return torch.cat(vectors, dim=0)

    def close(self):
        del self.model
        del self.tokenizer
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()


def cosine_pair(emb, a, b):
    ea, eb = emb.embed([emb.prepare_text(a), emb.prepare_text(b)], batch_size=2)
    return float(torch.dot(ea, eb).item())


def eval_pair_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: pairwise', unit='pair')):
        sim = cosine_pair(emb, str(r.story_text_a), str(r.story_text_b))
        rows.append({
            'metric': 'pairwise_gap',
            'task': 'eval_pair_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'pair_id', idx)),
            'dataset': getattr(r, 'dataset', None),
            'label': int(r.label),
            'score': sim,
        })
    return rows


def retrieval_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: retrieval', unit='query')):
        q = emb.prepare_text(str(r.query_text))
        opts = [emb.prepare_text(str(getattr(r, f'option_{i}_text'))) for i in range(5)]
        eq = emb.embed([q], batch_size=1).squeeze(0)
        eo = emb.embed(opts, batch_size=5)
        sims = (eo @ eq).numpy()
        pred = int(np.argmax(sims))
        gt = int(r.correct_option_index)
        rows.append({
            'metric': 'retrieval_accuracy',
            'task': 'retrieval_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'query_id', getattr(r, 'pair_id', idx))),
            'dataset': getattr(r, 'dataset', None),
            'correct': int(pred == gt),
            'pred': pred,
            'gold': gt,
        })
    return rows


def ndcg_at_5_from_ranks(sims, gold_ranks):
    rel = np.array([6 - int(x) for x in gold_ranks], dtype=np.float64)
    pred_order = np.argsort(-np.asarray(sims, dtype=np.float64))
    gains = rel[pred_order]
    discounts = 1.0 / np.log2(np.arange(2, 2 + len(gains), dtype=np.float64))
    dcg = float(np.sum(gains * discounts))
    ideal_order = np.argsort(-rel)
    idcg = float(np.sum(rel[ideal_order] * discounts))
    return float(dcg / idcg) if idcg > 0 else 0.0


def ranking_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: ranking', unit='query')):
        q = emb.prepare_text(str(r.query_text))
        opts = [emb.prepare_text(str(getattr(r, f'option_{i}_text'))) for i in range(5)]
        eq = emb.embed([q], batch_size=1).squeeze(0)
        eo = emb.embed(opts, batch_size=5)
        sims = (eo @ eq).numpy()
        ranks = [int(getattr(r, f'option_{i}_rank')) for i in range(5)]
        ndcg = ndcg_at_5_from_ranks(sims, ranks)
        rows.append({
            'metric': 'ranking_ndcg',
            'task': 'ranking_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'pair_id', idx)),
            'dataset': getattr(r, 'dataset', None),
            'ndcg': ndcg,
        })
    return rows


def synthetic_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: synthetic', unit='theme')):
        s12 = float(r.struct_score_12)
        s13 = float(r.struct_score_13)
        if s12 == s13:
            continue
        gt = 'story_2' if s12 > s13 else 'story_3'
        e = emb.embed([emb.prepare_text(str(r.story_1)), emb.prepare_text(str(r.story_2)), emb.prepare_text(str(r.story_3))], batch_size=3)
        sim12 = float(torch.dot(e[0], e[1]).item())
        sim13 = float(torch.dot(e[0], e[2]).item())
        pred = 'story_2' if sim12 > sim13 else 'story_3'
        rows.append({
            'metric': 'synthetic_accuracy',
            'task': 'synthetic_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'theme', idx)),
            'dataset': 'synthetic_v2',
            'correct': int(pred == gt),
            'pred': pred,
            'gold': gt,
        })
    return rows


def semeval_2026_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: semeval2026', unit='triplet')):
        e = emb.embed([
            emb.prepare_text(str(r.anchor_text)),
            emb.prepare_text(str(r.text_a)),
            emb.prepare_text(str(r.text_b)),
        ], batch_size=3)
        sim_a = float(torch.dot(e[0], e[1]).item())
        sim_b = float(torch.dot(e[0], e[2]).item())
        pred = bool(sim_a > sim_b)
        gt = parse_bool_label(r.text_a_is_closer)
        rows.append({
            'metric': 'semeval_2026_accuracy',
            'task': 'semeval_2026_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(idx),
            'dataset': 'semeval_2026_task_4',
            'correct': int(pred == gt),
            'pred': pred,
            'gold': gt,
        })
    return rows


def semeval_2022_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: semeval2022', unit='pair')):
        sim = cosine_pair(emb, str(r.article_1), str(r.article_2))
        rows.append({
            'metric': 'semeval_2022_spearman',
            'task': 'semeval_2022_correlation_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'pair_id', idx)),
            'dataset': 'semeval_2022_task_8',
            'score': sim,
            'human_NAR': float(r.NAR),
        })
    return rows


def load_eval_frames():
    frames = {}
    frames['eval_pair'] = pd.read_csv(PATHS['eval_pair'])
    frames['retrieval'] = pd.read_csv(PATHS['retrieval'])
    frames['ranking'] = pd.read_csv(PATHS['ranking'])
    frames['synthetic_v2'] = pd.read_csv(PATHS['synthetic_v2'])
    frames['semeval_2026'] = read_jsonl(PATHS['semeval_2026'])
    frames['semeval_2022'] = pd.read_csv(PATHS['semeval_2022'])
    return frames


def load_existing_llm_outputs():
    rows = []
    if not INCLUDE_EXISTING_LLM_OUTPUTS:
        return pd.DataFrame(rows)

    # Ranking GPT-5.4 outputs.
    p = EVAL_RESULTS_DIR / 'ranking_task_gpt54_predictions.csv'
    if p.exists():
        df = pd.read_csv(p)
        for r in df.itertuples(index=False):
            if pd.isna(getattr(r, 'ndcg_at_5')):
                continue
            rows.append({
                'metric': 'ranking_ndcg', 'task': 'ranking_task', 'model': str(r.model), 'model_type': 'llm_baseline',
                'example_id': str(r.pair_id), 'dataset': getattr(r, 'dataset', None), 'ndcg': float(r.ndcg_at_5),
            })

    # Synthetic GPT-5.4 outputs.
    p = EVAL_RESULTS_DIR / 'synthetic_anchor_eval_gpt54_predictions.csv'
    if p.exists():
        df = pd.read_csv(p)
        for idx, r in enumerate(df.itertuples(index=False)):
            rows.append({
                'metric': 'synthetic_accuracy', 'task': 'synthetic_task', 'model': 'gpt-5.4', 'model_type': 'llm_baseline',
                'example_id': str(getattr(r, 'theme', idx)), 'dataset': 'synthetic_v2', 'correct': int(bool(r.correct)),
                'pred': str(r.pred_higher_vs_story_1), 'gold': str(r.gt_higher_vs_story_1),
            })

    # SemEval 2026 GPT-5.4 outputs.
    p = EVAL_RESULTS_DIR / 'semeval_2026_gpt54_predictions.csv'
    if p.exists():
        df = pd.read_csv(p)
        for r in df.itertuples(index=False):
            rows.append({
                'metric': 'semeval_2026_accuracy', 'task': 'semeval_2026_task', 'model': str(r.model), 'model_type': 'llm_baseline',
                'example_id': str(r.row_index), 'dataset': 'semeval_2026_task_4', 'correct': int(bool(r.correct)),
                'pred': str(r.predicted_closer), 'gold': bool(r.gold_text_a_is_closer),
            })

    # SemEval 2022 GPT-5.4 NAR scores.
    p = EVAL_RESULTS_DIR / 'semeval_2022_task8_gpt54_nar_scores.csv'
    if p.exists():
        df = pd.read_csv(p)
        for r in df.itertuples(index=False):
            if pd.isna(getattr(r, 'llm_nar_score_1_to_4')):
                continue
            rows.append({
                'metric': 'semeval_2022_spearman', 'task': 'semeval_2022_correlation_task', 'model': str(r.model), 'model_type': 'llm_baseline',
                'example_id': str(r.pair_id), 'dataset': 'semeval_2022_task_8',
                'score': float(r.llm_nar_score_1_to_4), 'human_NAR': float(r.human_NAR),
            })

    return pd.DataFrame(rows)


def _save_per_example_outputs(per_example):
    try:
        per_example.to_parquet(PER_EXAMPLE_PARQUET, index=False)
        print('Saved:', PER_EXAMPLE_PARQUET)
    except Exception as e:
        print('[warn] Could not save parquet:', e)
    per_example.to_csv(PER_EXAMPLE_CSV, index=False)
    print('Saved:', PER_EXAMPLE_CSV)


def _load_cached_per_example_outputs():
    if PER_EXAMPLE_PARQUET.exists():
        print('Loading cached per-example outputs:', PER_EXAMPLE_PARQUET)
        return pd.read_parquet(PER_EXAMPLE_PARQUET)
    if PER_EXAMPLE_CSV.exists():
        print('Loading cached per-example outputs:', PER_EXAMPLE_CSV)
        return pd.read_csv(PER_EXAMPLE_CSV)
    return pd.DataFrame()


def _run_all_tasks_for_spec(spec, frames):
    print('\n=== Evaluating', spec['model'], '===')
    emb = Embedder(
        spec['path'],
        max_length=spec['max_length'],
        use_bfloat16=spec['use_bfloat16'],
        use_e5_query_prefix=spec['use_e5_query_prefix'],
    )
    rows = []
    rows.extend(eval_pair_outputs(frames['eval_pair'], emb, spec['model'], spec['model_type']))
    rows.extend(retrieval_outputs(frames['retrieval'], emb, spec['model'], spec['model_type']))
    rows.extend(ranking_outputs(frames['ranking'], emb, spec['model'], spec['model_type']))
    rows.extend(synthetic_outputs(frames['synthetic_v2'], emb, spec['model'], spec['model_type']))
    rows.extend(semeval_2026_outputs(frames['semeval_2026'], emb, spec['model'], spec['model_type']))
    rows.extend(semeval_2022_outputs(frames['semeval_2022'], emb, spec['model'], spec['model_type']))
    emb.close()
    return rows


def build_embedding_per_example_outputs(force_recompute=False):
    expected_models = [spec['model'] for spec in MODEL_SPECS]

    if force_recompute:
        per_example = pd.DataFrame()
        missing_specs = MODEL_SPECS
        print('FORCE_RECOMPUTE=True, rebuilding all checkpoint/baseline outputs.')
    else:
        per_example = _load_cached_per_example_outputs()
        if len(per_example):
            # This notebook path is for embedding baselines only. Drop old GPT rows if
            # they were cached by an earlier version of the notebook.
            per_example = per_example[per_example['model'].isin(expected_models)].copy()
            present_models = sorted(per_example['model'].dropna().unique())
            print('Present cached embedding models:', present_models)
        else:
            present_models = []
            print('No cached embedding outputs found.')

        missing_specs = [spec for spec in MODEL_SPECS if spec['model'] not in set(present_models)]

    if missing_specs:
        print('Missing models to compute:', [spec['model'] for spec in missing_specs])
        frames = load_eval_frames()
        new_rows = []
        for spec in missing_specs:
            new_rows.extend(_run_all_tasks_for_spec(spec, frames))

        new_df = pd.DataFrame(new_rows)
        per_example = pd.concat([per_example, new_df], ignore_index=True, sort=False) if len(per_example) else new_df
        per_example = per_example[per_example['model'].isin(expected_models)].copy()
        _save_per_example_outputs(per_example)
    else:
        print('All checkpoint/baseline model outputs are already present in cache.')

    missing_after = sorted(set(expected_models) - set(per_example['model'].dropna().unique()))
    if missing_after:
        raise RuntimeError(f'Missing expected model outputs after reconstruction: {missing_after}')

    return per_example.reset_index(drop=True)


per_example_df = build_embedding_per_example_outputs(force_recompute=FORCE_RECOMPUTE)
print('Rows:', len(per_example_df))
display(per_example_df.groupby(['metric', 'model', 'model_type']).size().reset_index(name='n').sort_values(['metric', 'model_type', 'model']))


No cached embedding outputs found.
Missing models to compute: ['E5-Mistral+MSE', 'bge_base', 'e5_untrained', 'roberta_base', 'story_emb']

=== Evaluating E5-Mistral+MSE ===


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.45it/s]


[Embedder] /scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss/epoch_01_mse_loss | device=cuda | max_length=1024 | e5_prefix=True


E5-Mistral+MSE: semeval2022: 100%|██████████| 90/90 [00:24<00:00,  3.63pair/s]



=== Evaluating bge_base ===
[Embedder] BAAI/bge-base-en-v1.5 | device=cuda | max_length=512 | e5_prefix=False


...

Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.49s/it]


[Embedder] uhhlt/story-emb | device=cuda | max_length=1024 | e5_prefix=False


story_emb: semeval2022: 100%|██████████| 90/90 [00:20<00:00,  4.37pair/s]


[warn] Could not save parquet: ("Could not convert 'story_2' with type str: tried to convert to int64", 'Conversion failed for column pred with type object')
Saved: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_results/statistical_testing/embedding_per_example_outputs_epoch02_mse_plus_baselines.csv
Rows: 4545


,metric,model,model_type,n
0,pairwise_gap,E5-Mistral+MSE,checkpoint,200
1,pairwise_gap,bge_base,embedding_baseline,200
2,pairwise_gap,e5_untrained,embedding_baseline,200
3,pairwise_gap,roberta_base,embedding_baseline,200
4,pairwise_gap,story_emb,embedding_baseline,200
5,ranking_ndcg,E5-Mistral+MSE,checkpoint,199
6,ranking_ndcg,bge_base,embedding_baseline,199
7,ranking_ndcg,e5_untrained,embedding_baseline,199
8,ranking_ndcg,roberta_base,embedding_baseline,199
9,ranking_ndcg,story_emb,embedding_baseline,199


## Bootstrap Confidence Intervals

For each model and metric, this cell computes a 95\% percentile bootstrap confidence interval using 10,000 resamples with replacement. Resampling is done over whole examples: one pair for pairwise separation and SemEval 2022, one query for retrieval/ranking, one theme for synthetic, and one triplet for SemEval 2026.

In [21]:
rng = np.random.default_rng(RANDOM_SEED)

METRIC_DISPLAY_NAMES = {
    'pairwise_gap': 'Pairwise separation gap',
    'retrieval_accuracy': 'Retrieval accuracy',
    'ranking_ndcg': 'Ranking nDCG',
    'synthetic_accuracy': 'Synthetic accuracy',
    'semeval_2026_accuracy': 'SemEval 2026 accuracy',
    'semeval_2022_spearman': 'SemEval 2022 Spearman',
}


def metric_point_estimate(df):
    metric = df['metric'].iloc[0]
    if metric == 'pairwise_gap':
        pos = df.loc[df['label'].astype(int) == 1, 'score'].astype(float)
        neg = df.loc[df['label'].astype(int) == 0, 'score'].astype(float)
        return float(pos.mean() - neg.mean())
    if metric in {'retrieval_accuracy', 'synthetic_accuracy', 'semeval_2026_accuracy'}:
        return float(df['correct'].astype(float).mean())
    if metric == 'ranking_ndcg':
        return float(df['ndcg'].astype(float).mean())
    if metric == 'semeval_2022_spearman':
        if len(df) < 3:
            return np.nan
        return float(stats.spearmanr(df['score'].astype(float), df['human_NAR'].astype(float), nan_policy='omit').statistic)
    raise ValueError(f'Unknown metric: {metric}')


def bootstrap_ci(df, stat_fn, n_boot=N_BOOT, alpha=ALPHA, seed=RANDOM_SEED):
    local_rng = np.random.default_rng(seed)
    n = len(df)
    if n == 0:
        return np.nan, np.nan
    vals = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        idx = local_rng.integers(0, n, size=n)
        vals[b] = stat_fn(df.iloc[idx])
    return tuple(np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)]))


point_rows = []
for (metric, model), g in tqdm(per_example_df.groupby(['metric', 'model']), desc='bootstrap CIs', unit='model-metric'):
    g = g.dropna(how='all').reset_index(drop=True)
    point = metric_point_estimate(g)
    ci_low, ci_high = bootstrap_ci(g, metric_point_estimate, seed=abs(hash((metric, model, RANDOM_SEED))) % (2**32))
    point_rows.append({
        'metric': metric,
        'metric_name': METRIC_DISPLAY_NAMES.get(metric, metric),
        'model': model,
        'model_type': g['model_type'].iloc[0] if 'model_type' in g.columns else None,
        'n_examples': int(len(g)),
        'point_estimate': point,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'ci_95': f'[{ci_low:.4f}, {ci_high:.4f}]',
    })

metric_ci_df = pd.DataFrame(point_rows).sort_values(['metric', 'model_type', 'model']).reset_index(drop=True)
metric_ci_df.to_csv(POINT_TABLE_PATH, index=False)
print('Saved:', POINT_TABLE_PATH)
display(metric_ci_df)

bootstrap CIs: 100%|██████████| 30/30 [01:41<00:00,  3.37s/model-metric]

Saved: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_results/statistical_testing/metric_bootstrap_ci_table.csv


,metric,metric_name,model,model_type,n_examples,point_estimate,ci_low,ci_high,ci_95
0,pairwise_gap,Pairwise separation gap,E5-Mistral+MSE,checkpoint,200,0.426855,0.394343,0.459372,"[0.3943, 0.4594]"
1,pairwise_gap,Pairwise separation gap,bge_base,embedding_baseline,200,0.146310,0.128626,0.164758,"[0.1286, 0.1648]"
2,pairwise_gap,Pairwise separation gap,e5_untrained,embedding_baseline,200,0.162377,0.149413,0.175765,"[0.1494, 0.1758]"
3,pairwise_gap,Pairwise separation gap,roberta_base,embedding_baseline,200,0.006863,-0.000583,0.014262,"[-0.0006, 0.0143]"
4,pairwise_gap,Pairwise separation gap,story_emb,embedding_baseline,200,0.171582,0.156910,0.186482,"[0.1569, 0.1865]"
5,ranking_ndcg,Ranking nDCG,E5-Mistral+MSE,checkpoint,199,0.978112,0.975187,0.981055,"[0.9752, 0.9811]"
6,ranking_ndcg,Ranking nDCG,bge_base,embedding_baseline,199,0.961578,0.955202,0.967423,"[0.9552, 0.9674]"
7,ranking_ndcg,Ranking nDCG,e5_untrained,embedding_baseline,199,0.975272,0.971667,0.978537,"[0.9717, 0.9785]"
8,ranking_ndcg,Ranking nDCG,roberta_base,embedding_baseline,199,0.913187,0.902858,0.923171,"[0.9029, 0.9232]"
9,ranking_ndcg,Ranking nDCG,story_emb,embedding_baseline,199,0.975525,0.971618,0.979045,"[0.9716, 0.9790]"


## Paired Significance Tests

This cell compares `E5-Mistral+MSE` against each available baseline on the same examples.

- Pairwise separation uses a paired bootstrap over the difference in positive-minus-negative gaps.
- Retrieval, synthetic, and SemEval 2026 accuracy use McNemar's test on paired correctness indicators.
- Ranking uses Wilcoxon signed-rank tests over paired per-query nDCG values.
- SemEval 2022 uses Steiger's dependent-correlation test, applied to rank-transformed variables so that the comparison matches Spearman correlation.

In [22]:
def align_model_rows(df, metric, model_a, model_b):
    a = df[(df['metric'] == metric) & (df['model'] == model_a)].copy()
    b = df[(df['metric'] == metric) & (df['model'] == model_b)].copy()
    common = sorted(set(a['example_id'].astype(str)) & set(b['example_id'].astype(str)))
    a = a[a['example_id'].astype(str).isin(common)].sort_values('example_id').reset_index(drop=True)
    b = b[b['example_id'].astype(str).isin(common)].sort_values('example_id').reset_index(drop=True)
    if len(a) != len(b):
        raise RuntimeError(f'Alignment failed for {metric}: {model_a} vs {model_b}')
    return a, b


def paired_bootstrap_gap(our_df, base_df, n_boot=N_BOOT, alpha=ALPHA, seed=RANDOM_SEED):
    merged = our_df[['example_id', 'label', 'score']].merge(
        base_df[['example_id', 'label', 'score']],
        on=['example_id', 'label'],
        suffixes=('_our', '_baseline'),
    )
    if len(merged) == 0:
        return np.nan, np.nan, np.nan, np.nan

    def gap_diff(x):
        our_pos = x.loc[x['label'].astype(int) == 1, 'score_our'].astype(float)
        our_neg = x.loc[x['label'].astype(int) == 0, 'score_our'].astype(float)
        base_pos = x.loc[x['label'].astype(int) == 1, 'score_baseline'].astype(float)
        base_neg = x.loc[x['label'].astype(int) == 0, 'score_baseline'].astype(float)
        return float((our_pos.mean() - our_neg.mean()) - (base_pos.mean() - base_neg.mean()))

    observed = gap_diff(merged)
    local_rng = np.random.default_rng(seed)
    vals = np.empty(n_boot, dtype=np.float64)
    n = len(merged)
    for i in range(n_boot):
        vals[i] = gap_diff(merged.iloc[local_rng.integers(0, n, size=n)])
    ci_low, ci_high = np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    p_boot = 2 * min(np.mean(vals <= 0), np.mean(vals >= 0))
    p_boot = float(min(1.0, p_boot))
    return observed, float(ci_low), float(ci_high), p_boot


def run_mcnemar(our_correct, base_correct):
    our = np.asarray(our_correct, dtype=bool)
    base = np.asarray(base_correct, dtype=bool)
    both_correct = int(np.sum(our & base))
    our_wrong_base_correct = int(np.sum((~our) & base))
    our_correct_base_wrong = int(np.sum(our & (~base)))
    both_wrong = int(np.sum((~our) & (~base)))
    table = [[both_correct, our_wrong_base_correct], [our_correct_base_wrong, both_wrong]]

    if statsmodels_mcnemar is not None:
        result = statsmodels_mcnemar(table, exact=True)
        return float(result.statistic), float(result.pvalue), table

    discordant = our_wrong_base_correct + our_correct_base_wrong
    if discordant == 0:
        return 0.0, 1.0, table
    p = stats.binomtest(min(our_wrong_base_correct, our_correct_base_wrong), n=discordant, p=0.5, alternative='two-sided').pvalue
    return float(min(our_wrong_base_correct, our_correct_base_wrong)), float(p), table


def run_wilcoxon(our_values, base_values):
    our = np.asarray(our_values, dtype=np.float64)
    base = np.asarray(base_values, dtype=np.float64)
    diffs = our - base
    if np.allclose(diffs, 0):
        return 0.0, 1.0
    res = stats.wilcoxon(our, base, zero_method='wilcox', alternative='two-sided')
    return float(res.statistic), float(res.pvalue)


def steiger_z_for_dependent_spearman(y, x1, x2):
    """Compare corr(y, x1) vs corr(y, x2) for overlapping dependent correlations.

    Spearman comparison is implemented by rank-transforming y, x1, and x2,
    then applying Steiger's dependent-correlation test to the Pearson
    correlations of the ranks.
    """
    y = np.asarray(y, dtype=np.float64)
    x1 = np.asarray(x1, dtype=np.float64)
    x2 = np.asarray(x2, dtype=np.float64)
    ok = np.isfinite(y) & np.isfinite(x1) & np.isfinite(x2)
    y, x1, x2 = y[ok], x1[ok], x2[ok]
    n = len(y)
    if n < 4:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    ry = stats.rankdata(y)
    rx1 = stats.rankdata(x1)
    rx2 = stats.rankdata(x2)
    r_y1 = float(np.corrcoef(ry, rx1)[0, 1])
    r_y2 = float(np.corrcoef(ry, rx2)[0, 1])
    r_12 = float(np.corrcoef(rx1, rx2)[0, 1])

    # Steiger/Williams test for two correlations sharing one variable.
    k = 1 - r_y1**2 - r_y2**2 - r_12**2 + 2 * r_y1 * r_y2 * r_12
    denom = math.sqrt(max(2 * k + ((r_y1 + r_y2) ** 2 / 4.0) * ((1 - r_12) ** 3), 1e-12))
    t_stat = (r_y1 - r_y2) * math.sqrt(max((n - 3) * (1 + r_12), 0.0)) / denom
    p_value = 2 * stats.t.sf(abs(t_stat), df=n - 3)
    return float(t_stat), float(p_value), r_y1, r_y2, r_12


test_rows = []
metrics = sorted(per_example_df['metric'].dropna().unique())
for metric in metrics:
    models = sorted(set(per_example_df.loc[per_example_df['metric'] == metric, 'model']) - {OUR_MODEL_NAME})
    for baseline in models:
        try:
            our, base = align_model_rows(per_example_df, metric, OUR_MODEL_NAME, baseline)
            if len(our) == 0:
                continue

            row = {
                'metric': metric,
                'metric_name': METRIC_DISPLAY_NAMES.get(metric, metric),
                'our_model': OUR_MODEL_NAME,
                'baseline_model': baseline,
                'n_paired_examples': int(len(our)),
            }

            if metric == 'pairwise_gap':
                diff, lo, hi, p = paired_bootstrap_gap(
                    our, base,
                    seed=abs(hash((metric, baseline, 'gap', RANDOM_SEED))) % (2**32),
                )
                row.update({
                    'test_name': 'paired bootstrap gap difference',
                    'test_statistic': diff,
                    'p_value': p,
                    'difference': diff,
                    'difference_ci_low': lo,
                    'difference_ci_high': hi,
                    'difference_ci_95': f'[{lo:.4f}, {hi:.4f}]',
                    'significant_alpha_0_05': bool(lo > 0 or hi < 0),
                })
            elif metric in {'retrieval_accuracy', 'synthetic_accuracy', 'semeval_2026_accuracy'}:
                stat, p, table = run_mcnemar(our['correct'].astype(int), base['correct'].astype(int))
                row.update({
                    'test_name': 'McNemar exact test',
                    'test_statistic': stat,
                    'p_value': p,
                    'mcnemar_table': str(table),
                    'significant_alpha_0_05': bool(p < ALPHA),
                })
            elif metric == 'ranking_ndcg':
                stat, p = run_wilcoxon(our['ndcg'].astype(float), base['ndcg'].astype(float))
                diff = float(our['ndcg'].astype(float).mean() - base['ndcg'].astype(float).mean())
                row.update({
                    'test_name': 'Wilcoxon signed-rank test',
                    'test_statistic': stat,
                    'p_value': p,
                    'difference': diff,
                    'significant_alpha_0_05': bool(p < ALPHA),
                })
            elif metric == 'semeval_2022_spearman':
                merged = our[['example_id', 'human_NAR', 'score']].merge(
                    base[['example_id', 'score']],
                    on='example_id',
                    suffixes=('_our', '_baseline'),
                )
                t_stat, p, r_our, r_base, r_between = steiger_z_for_dependent_spearman(
                    merged['human_NAR'], merged['score_our'], merged['score_baseline']
                )
                row.update({
                    'test_name': 'Steiger dependent Spearman test',
                    'test_statistic': t_stat,
                    'p_value': p,
                    'our_spearman': r_our,
                    'baseline_spearman': r_base,
                    'model_score_correlation': r_between,
                    'difference': float(r_our - r_base),
                    'significant_alpha_0_05': bool(p < ALPHA) if np.isfinite(p) else False,
                })
            else:
                continue
            test_rows.append(row)
        except Exception as e:
            warnings.warn(f'Failed {metric}: {OUR_MODEL_NAME} vs {baseline}: {e}')

test_df = pd.DataFrame(test_rows).sort_values(['metric', 'baseline_model']).reset_index(drop=True)
test_df.to_csv(TEST_TABLE_PATH, index=False)
print('Saved:', TEST_TABLE_PATH)
display(test_df)

Saved: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_results/statistical_testing/paired_significance_tests.csv


,metric,metric_name,our_model,baseline_model,n_paired_examples,test_name,test_statistic,p_value,difference,difference_ci_low,difference_ci_high,difference_ci_95,significant_alpha_0_05,mcnemar_table,our_spearman,baseline_spearman,model_score_correlation
0,pairwise_gap,Pairwise separation gap,E5-Mistral+MSE,bge_base,200,paired bootstrap gap difference,0.280545,0.000000e+00,0.280545,0.253612,0.306986,"[0.2536, 0.3070]",True,NaN,NaN,NaN,NaN
1,pairwise_gap,Pairwise separation gap,E5-Mistral+MSE,e5_untrained,200,paired bootstrap gap difference,0.264478,0.000000e+00,0.264478,0.240653,0.288396,"[0.2407, 0.2884]",True,NaN,NaN,NaN,NaN
2,pairwise_gap,Pairwise separation gap,E5-Mistral+MSE,roberta_base,200,paired bootstrap gap difference,0.419992,0.000000e+00,0.419992,0.389754,0.450516,"[0.3898, 0.4505]",True,NaN,NaN,NaN,NaN
3,pairwise_gap,Pairwise separation gap,E5-Mistral+MSE,story_emb,200,paired bootstrap gap difference,0.255273,0.000000e+00,0.255273,0.231334,0.279138,"[0.2313, 0.2791]",True,NaN,NaN,NaN,NaN
4,ranking_ndcg,Ranking nDCG,E5-Mistral+MSE,bge_base,199,Wilcoxon signed-rank test,4606.000000,2.586064e-06,0.016534,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN
5,ranking_ndcg,Ranking nDCG,E5-Mistral+MSE,e5_untrained,199,Wilcoxon signed-rank test,5432.500000,1.408763e-01,0.002839,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
6,ranking_ndcg,Ranking nDCG,E5-Mistral+MSE,roberta_base,199,Wilcoxon signed-rank test,1295.000000,1.221102e-24,0.064925,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN
7,ranking_ndcg,Ranking nDCG,E5-Mistral+MSE,story_emb,199,Wilcoxon signed-rank test,5337.500000,2.556230e-01,0.002587,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
8,retrieval_accuracy,Retrieval accuracy,E5-Mistral+MSE,bge_base,200,McNemar exact test,2.000000,2.212524e-04,NaN,NaN,NaN,NaN,True,"[[175, 2], [19, 4]]",NaN,NaN,NaN
9,retrieval_accuracy,Retrieval accuracy,E5-Mistral+MSE,e5_untrained,200,McNemar exact test,3.000000,1.000000e+00,NaN,NaN,NaN,NaN,False,"[[191, 3], [3, 3]]",NaN,NaN,NaN


## Final Reporting Table

This table combines each model's point estimate and bootstrap confidence interval with the paired significance result against `E5-Mistral+MSE` whenever that comparison is defined. For the primary model itself, test columns are left blank because it is the reference system.

In [10]:
summary = metric_ci_df.copy()
summary['point_with_ci'] = summary.apply(
    lambda r: f"{r['point_estimate']:.4f} {r['ci_95']}", axis=1
)

comparison_cols = [
    'metric', 'baseline_model', 'test_name', 'test_statistic', 'p_value',
    'difference', 'difference_ci_95', 'significant_alpha_0_05',
    'n_paired_examples', 'mcnemar_table',
]
comparison_cols = [c for c in comparison_cols if c in test_df.columns]
comp = test_df[comparison_cols].rename(columns={'baseline_model': 'model'}) if len(test_df) else pd.DataFrame(columns=['metric', 'model'])

summary = summary.merge(comp, on=['metric', 'model'], how='left')
summary = summary[[
    'metric_name', 'metric', 'model_type', 'model', 'n_examples',
    'point_estimate', 'ci_95', 'test_name', 'test_statistic', 'p_value',
    'difference', 'difference_ci_95', 'significant_alpha_0_05', 'n_paired_examples'
] if 'n_paired_examples' in summary.columns else [
    'metric_name', 'metric', 'model_type', 'model', 'n_examples',
    'point_estimate', 'ci_95', 'test_name', 'test_statistic', 'p_value',
    'difference', 'difference_ci_95', 'significant_alpha_0_05'
]]
summary = summary.sort_values(['metric', 'model_type', 'model']).reset_index(drop=True)
summary.to_csv(SUMMARY_TABLE_PATH, index=False)
print('Saved:', SUMMARY_TABLE_PATH)
display(summary)

KeyError: "['difference_ci_95'] not in index"

## Notes for Paper Reporting

Use `metric_bootstrap_ci_table.csv` for confidence intervals around point estimates and `paired_significance_tests.csv` for direct model-vs-baseline tests. The combined `statistical_testing_summary.csv` is convenient for paper tables.

Interpretation conventions:

- For pairwise separation, positive differences mean `E5-Mistral+MSE` has a larger positive-minus-negative cosine gap than the baseline.
- For accuracy and nDCG, positive differences mean `E5-Mistral+MSE` performs better on average.
- For SemEval 2022, Spearman correlation is the primary metric because NAR is ordinal. The Steiger test is applied after rank-transforming model scores and human NAR labels.

In [11]:
# Paired significance tests: E5-Mistral+MSE vs all non-LLM embedding baselines across all tasks.
#
# Expected output:
# 5 baselines x 6 metrics = 30 rows, assuming all baseline per-example outputs are present.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import math

try:
    from statsmodels.stats.contingency_tables import mcnemar as statsmodels_mcnemar
except Exception:
    statsmodels_mcnemar = None

OUR_MODEL = "E5-Mistral+MSE"
BASELINE_MODELS = [
    "bge_base",
    "e5_untrained",
    "roberta_base",
    "story_emb",
    "gpt-5.4",
]

METRIC_DISPLAY_NAMES = {
    "pairwise_gap": "Pairwise separation gap",
    "retrieval_accuracy": "Retrieval accuracy",
    "ranking_ndcg": "Ranking nDCG",
    "synthetic_accuracy": "Synthetic accuracy",
    "semeval_2026_accuracy": "SemEval 2026 accuracy",
    "semeval_2022_spearman": "SemEval 2022 Spearman",
}

TASK_METRICS = [
    "pairwise_gap",
    "retrieval_accuracy",
    "ranking_ndcg",
    "synthetic_accuracy",
    "semeval_2026_accuracy",
    "semeval_2022_spearman",
]

N_BOOT = 10_000
ALPHA = 0.05
RANDOM_SEED = 42

OUT_DIR = PROJECT_ROOT / "data" / "eval_results" / "statistical_testing"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "all_baseline_paired_significance_tests.csv"


def _stable_seed(*parts, base_seed=RANDOM_SEED):
    # Python's built-in hash is salted per process, so use a stable hash for exact reruns.
    import hashlib
    s = "::".join(map(str, parts)) + f"::{base_seed}"
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16)


def _align_model_rows(df, metric, model_a, model_b):
    a = df[(df["metric"] == metric) & (df["model"] == model_a)].copy()
    b = df[(df["metric"] == metric) & (df["model"] == model_b)].copy()

    common = sorted(set(a["example_id"].astype(str)) & set(b["example_id"].astype(str)))
    a = a[a["example_id"].astype(str).isin(common)].sort_values("example_id").reset_index(drop=True)
    b = b[b["example_id"].astype(str).isin(common)].sort_values("example_id").reset_index(drop=True)

    if len(a) != len(b):
        raise RuntimeError(f"Alignment failed for {metric}: {model_a} vs {model_b}")

    return a, b


def _gap(df, score_col):
    pos = df.loc[df["label"].astype(int) == 1, score_col].astype(float)
    neg = df.loc[df["label"].astype(int) == 0, score_col].astype(float)
    return float(pos.mean() - neg.mean())


def paired_bootstrap_gap_test(our_df, base_df, n_boot=N_BOOT, alpha=ALPHA, seed=RANDOM_SEED):
    merged = our_df[["example_id", "label", "score"]].merge(
        base_df[["example_id", "label", "score"]],
        on=["example_id", "label"],
        suffixes=("_our", "_baseline"),
    )

    if merged.empty:
        return np.nan, np.nan, np.nan, np.nan

    def stat_fn(x):
        return _gap(x, "score_our") - _gap(x, "score_baseline")

    observed = stat_fn(merged)
    rng = np.random.default_rng(seed)
    n = len(merged)

    boot = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        sample_idx = rng.integers(0, n, size=n)
        boot[i] = stat_fn(merged.iloc[sample_idx])

    ci_low, ci_high = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])

    # Bootstrap two-sided p-value around zero.
    p_value = 2 * min(np.mean(boot <= 0), np.mean(boot >= 0))
    p_value = float(min(1.0, p_value))

    return float(observed), float(ci_low), float(ci_high), p_value


def mcnemar_exact_test(our_correct, base_correct):
    our = np.asarray(our_correct, dtype=bool)
    base = np.asarray(base_correct, dtype=bool)

    both_correct = int(np.sum(our & base))
    our_wrong_base_correct = int(np.sum((~our) & base))
    our_correct_base_wrong = int(np.sum(our & (~base)))
    both_wrong = int(np.sum((~our) & (~base)))

    table = [
        [both_correct, our_wrong_base_correct],
        [our_correct_base_wrong, both_wrong],
    ]

    if statsmodels_mcnemar is not None:
        res = statsmodels_mcnemar(table, exact=True)
        return float(res.statistic), float(res.pvalue), table

    discordant = our_wrong_base_correct + our_correct_base_wrong
    if discordant == 0:
        return 0.0, 1.0, table

    p_value = stats.binomtest(
        min(our_wrong_base_correct, our_correct_base_wrong),
        n=discordant,
        p=0.5,
        alternative="two-sided",
    ).pvalue
    return float(min(our_wrong_base_correct, our_correct_base_wrong)), float(p_value), table


def wilcoxon_signed_rank_test(our_values, base_values):
    our = np.asarray(our_values, dtype=np.float64)
    base = np.asarray(base_values, dtype=np.float64)

    if np.allclose(our - base, 0):
        return 0.0, 1.0

    res = stats.wilcoxon(our, base, zero_method="wilcox", alternative="two-sided")
    return float(res.statistic), float(res.pvalue)


def steiger_dependent_spearman_test(y, x_our, x_base):
    """
    Compare two dependent Spearman correlations:
        corr(y, x_our) vs corr(y, x_base)

    Implementation: rank-transform all three variables, then apply Steiger/Williams
    overlapping-correlation test to Pearson correlations of the ranks.
    """
    y = np.asarray(y, dtype=np.float64)
    x_our = np.asarray(x_our, dtype=np.float64)
    x_base = np.asarray(x_base, dtype=np.float64)

    ok = np.isfinite(y) & np.isfinite(x_our) & np.isfinite(x_base)
    y = y[ok]
    x_our = x_our[ok]
    x_base = x_base[ok]

    n = len(y)
    if n < 4:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    ry = stats.rankdata(y)
    rx_our = stats.rankdata(x_our)
    rx_base = stats.rankdata(x_base)

    r_y_our = float(np.corrcoef(ry, rx_our)[0, 1])
    r_y_base = float(np.corrcoef(ry, rx_base)[0, 1])
    r_our_base = float(np.corrcoef(rx_our, rx_base)[0, 1])

    # Steiger/Williams test for two correlations sharing one variable.
    k = (
        1
        - r_y_our**2
        - r_y_base**2
        - r_our_base**2
        + 2 * r_y_our * r_y_base * r_our_base
    )
    denom = math.sqrt(
        max(
            2 * k + ((r_y_our + r_y_base) ** 2 / 4.0) * ((1 - r_our_base) ** 3),
            1e-12,
        )
    )
    t_stat = (r_y_our - r_y_base) * math.sqrt(max((n - 3) * (1 + r_our_base), 0.0)) / denom
    p_value = 2 * stats.t.sf(abs(t_stat), df=n - 3)

    return float(t_stat), float(p_value), r_y_our, r_y_base, r_our_base


available_models = sorted(per_example_df["model"].dropna().unique())
missing_baselines = [m for m in BASELINE_MODELS if m not in available_models]

if missing_baselines:
    print("Warning: these requested baselines are not present in per_example_df and will be skipped:")
    print(missing_baselines)

rows = []

for baseline in BASELINE_MODELS:
    if baseline not in available_models:
        continue

    for metric in TASK_METRICS:
        if baseline == "gpt-5.4" and metric in {"pairwise_gap", "retrieval_accuracy"}:
            # GPT-5.4 does not have the same pairwise-gap/retrieval outputs in the current cache.
            continue

        try:
            our_df, base_df = _align_model_rows(per_example_df, metric, OUR_MODEL, baseline)
            if len(our_df) == 0:
                print(f"Skipping empty alignment: {metric} | {OUR_MODEL} vs {baseline}")
                continue

            row = {
                "metric": metric,
                "metric_name": METRIC_DISPLAY_NAMES.get(metric, metric),
                "our_model": OUR_MODEL,
                "baseline_model": baseline,
                "n_paired_examples": int(len(our_df)),
                "test_name": None,
                "test_statistic": np.nan,
                "p_value": np.nan,
                "difference": np.nan,
                "difference_ci_low": np.nan,
                "difference_ci_high": np.nan,
                "difference_ci_95": None,
                "significant_alpha_0_05": False,
                "our_spearman": np.nan,
                "baseline_spearman": np.nan,
                "model_score_correlation": np.nan,
                "mcnemar_table": None,
            }

            if metric == "pairwise_gap":
                diff, lo, hi, p = paired_bootstrap_gap_test(
                    our_df,
                    base_df,
                    seed=_stable_seed(metric, baseline, "paired_bootstrap_gap"),
                )
                row.update({
                    "test_name": "paired bootstrap gap difference",
                    "test_statistic": diff,
                    "p_value": p,
                    "difference": diff,
                    "difference_ci_low": lo,
                    "difference_ci_high": hi,
                    "difference_ci_95": f"[{lo:.4f}, {hi:.4f}]",
                    "significant_alpha_0_05": bool(lo > 0 or hi < 0),
                })

            elif metric in {"retrieval_accuracy", "synthetic_accuracy", "semeval_2026_accuracy"}:
                stat, p, table = mcnemar_exact_test(
                    our_df["correct"].astype(int),
                    base_df["correct"].astype(int),
                )
                diff = float(our_df["correct"].astype(float).mean() - base_df["correct"].astype(float).mean())
                row.update({
                    "test_name": "McNemar exact test",
                    "test_statistic": stat,
                    "p_value": p,
                    "difference": diff,
                    "significant_alpha_0_05": bool(p < ALPHA),
                    "mcnemar_table": str(table),
                })

            elif metric == "ranking_ndcg":
                stat, p = wilcoxon_signed_rank_test(
                    our_df["ndcg"].astype(float),
                    base_df["ndcg"].astype(float),
                )
                diff = float(our_df["ndcg"].astype(float).mean() - base_df["ndcg"].astype(float).mean())
                row.update({
                    "test_name": "Wilcoxon signed-rank test",
                    "test_statistic": stat,
                    "p_value": p,
                    "difference": diff,
                    "significant_alpha_0_05": bool(p < ALPHA),
                })

            elif metric == "semeval_2022_spearman":
                merged = our_df[["example_id", "human_NAR", "score"]].merge(
                    base_df[["example_id", "score"]],
                    on="example_id",
                    suffixes=("_our", "_baseline"),
                )

                stat, p, our_rho, base_rho, model_corr = steiger_dependent_spearman_test(
                    merged["human_NAR"],
                    merged["score_our"],
                    merged["score_baseline"],
                )

                row.update({
                    "test_name": "Steiger dependent Spearman test",
                    "test_statistic": stat,
                    "p_value": p,
                    "difference": float(our_rho - base_rho),
                    "significant_alpha_0_05": bool(p < ALPHA) if np.isfinite(p) else False,
                    "our_spearman": our_rho,
                    "baseline_spearman": base_rho,
                    "model_score_correlation": model_corr,
                })

            rows.append(row)

        except Exception as e:
            print(f"Failed: {metric} | {OUR_MODEL} vs {baseline}: {e}")

all_baseline_tests_df = pd.DataFrame(rows)

expected_rows_without_gpt_pair_retrieval = 4 * 6 + 1 * 4
print("Rows produced:", len(all_baseline_tests_df))
print("Expected if 4 embedding baselines + GPT-5.4 available:", expected_rows_without_gpt_pair_retrieval)
print("Expected if only 4 embedding baselines:", 4 * 6)

synthetic_n = all_baseline_tests_df[
    all_baseline_tests_df["metric"] == "synthetic_accuracy"
][["baseline_model", "n_paired_examples"]].drop_duplicates()

print("\nSynthetic paired example counts:")
display(synthetic_n)

if len(synthetic_n):
    unique_ns = sorted(synthetic_n["n_paired_examples"].unique())
    print("Unique synthetic n_paired_examples:", unique_ns)
    if unique_ns == [20]:
        print("Synthetic task is using 20 examples.")
    elif unique_ns == [100]:
        print("Synthetic task is using 100 examples.")
    else:
        print("Synthetic task has mixed/unexpected paired counts:", unique_ns)

schema_cols = [
    "metric",
    "metric_name",
    "our_model",
    "baseline_model",
    "n_paired_examples",
    "test_name",
    "test_statistic",
    "p_value",
    "difference",
    "significant_alpha_0_05",
    "our_spearman",
    "baseline_spearman",
    "model_score_correlation",
    "mcnemar_table",
    "difference_ci_low",
    "difference_ci_high",
    "difference_ci_95",
]

schema_cols = [c for c in schema_cols if c in all_baseline_tests_df.columns]
all_baseline_tests_df = all_baseline_tests_df[schema_cols].sort_values(
    ["metric", "baseline_model"]
).reset_index(drop=True)

all_baseline_tests_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)

display(all_baseline_tests_df)

['bge_base', 'e5_untrained', 'roberta_base', 'story_emb']
Rows produced: 4
Expected if 4 embedding baselines + GPT-5.4 available: 28
Expected if only 4 embedding baselines: 24

Synthetic paired example counts:


,baseline_model,n_paired_examples
1,gpt-5.4,20


Unique synthetic n_paired_examples: [np.int64(20)]
Synthetic task is using 20 examples.

Saved: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/eval_results/statistical_testing/all_baseline_paired_significance_tests.csv


,metric,metric_name,our_model,baseline_model,n_paired_examples,test_name,test_statistic,p_value,difference,significant_alpha_0_05,our_spearman,baseline_spearman,model_score_correlation,mcnemar_table,difference_ci_low,difference_ci_high,difference_ci_95
0,ranking_ndcg,Ranking nDCG,E5-Mistral+MSE,gpt-5.4,199,Wilcoxon signed-rank test,3051.000000,3.216969e-11,-0.011176,True,NaN,NaN,NaN,None,NaN,NaN,None
1,semeval_2022_spearman,SemEval 2022 Spearman,E5-Mistral+MSE,gpt-5.4,90,Steiger dependent Spearman test,0.840515,4.029240e-01,0.031577,False,-0.72342,-0.754997,0.849988,None,NaN,NaN,None
2,semeval_2026_accuracy,SemEval 2026 accuracy,E5-Mistral+MSE,gpt-5.4,200,McNemar exact test,37.000000,5.764306e-01,-0.030000,False,NaN,NaN,NaN,"[[89, 43], [37, 31]]",NaN,NaN,None
3,synthetic_accuracy,Synthetic accuracy,E5-Mistral+MSE,gpt-5.4,20,McNemar exact test,5.000000,2.101135e-01,-0.300000,False,NaN,NaN,NaN,"[[4, 11], [5, 0]]",NaN,NaN,None
